# Tarteel Whisper Fine-Tuning — Surah 113 Al-Falaq + 114 An-Nas (Multi-Reciter Dataset)

Ye notebook `tarteel-ai/whisper-base-ar-quran` ko humare **real Quran dataset** par fine-tune karta hai — jo humare Quran Audio Segmenter tool se generate hua hai. Ab **do surahs** ka data shamil hai:

- **Surah 114 An-Nas** — 11 reciters, har qari ke 12 **phrase-level** reference segments (basmalah + 6 ayahs)
- **Surah 113 Al-Falaq** — 10 reciters, har qari ke **21 word-level** reference segments (client ke word-level spec ke mutabiq, 5 ayaat, koi basmalah nahi)

**Original demo sirf 1 reciter ke 12 clips pe based tha (12-sample toy dataset). Aapke `output` folder mein actually:**

- **12 unique reciters** (Muhammad Siddiq Al-Minshawi, Abdul Basit, Saad Al-Ghamdi, Ibrahim Al-Akhdar, Hatem Fareed Al-Waer, Al-Shatri, Khalifa Al-Tunaiji, Salah Bukhatir, Saud Al-Shuraim, Ahmed El-Agamy, Khalid Al-Jalil, + Bandar Balila)
- Har reciter ke paas surah-wise reference-matched segments (113: word-level, 114: phrase-level)
- Total **330 labeled audio segments** (210 × Al-Falaq + 122 × An-Nas)
- Har segment ka exact Arabic transcription reference JSON (`*_reference_segments.json`) mein already maujood hai — humein manually manifest banane ki zaroorat nahi, seedha JSON se parse karenge

**Ye ab ek chhota-sa demo nahi raha — ye ek proper multi-speaker fine-tuning dataset hai**, isliye is notebook mein ye improvements kiye gaye hain:

1. Manifest ab saare reciters ke (dono surahs ke) reference JSON se **automatically** parse hota hai (hardcoded 12-line list ki jagah)
2. **2 reciters completely held out** hote hain validation ke liye — taake hum asal mein test kar sakein ke model naye/unseen reciters (unheard voices) par kaisa perform karta hai, sirf memorization nahi
3. Training thoda zyada steps ke saath (dono surahs ka data ab), aur periodic evaluation on held-out reciters
4. Final evaluation train-reciters vs held-out-reciters dono par alag dikhati hai — taake overfitting vs real generalization clearly nazar aaye

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**
2. Cells order mein chalayein
3. Cell 3 mein apna `output.zip` (jo aapke Quran Audio Segmenter ne banaya) upload karein
4. Training complete hone ke baad, fine-tuned model download/test kar sakte hain

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Dependencies install karein

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install transformers datasets torch librosa soundfile jiwer accelerate evaluate
print('Dependencies installed.')

## 3. Dataset upload karein

Apna `output.zip` yahan upload karein (Quran Audio Segmenter ka poora output — sab reciter folders, har ek ke andar `segments/` aur `*_reference_segments.json`).

In [ ]:
from google.colab import files
import zipfile
import os

DATASET_DIR = '/content/quran_dataset'
os.makedirs(DATASET_DIR, exist_ok=True)

print('output.zip upload karein:')
uploaded = files.upload()

for fname in uploaded.keys():
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall(DATASET_DIR)
        os.remove(fname)
        print(f'Extracted {fname} into {DATASET_DIR}')

print('\nTop-level extracted folders:')
for root, dirs, fs in os.walk(DATASET_DIR):
    depth = root.replace(DATASET_DIR, '').count(os.sep)
    if depth <= 1:
        for d in dirs:
            print(' ', os.path.join(root, d))


## 4. Manifest banayein — saare reciters ke reference JSON se (automatic)

Pehle hardcoded 12-line manifest thi. Ab hum **saare reciter folders** dhoondh kar unke `*_reference_segments.json` (dono surahs 113 + 114) se transcription + audio path parse karte hain. Sirf wahi segments include hote hain jinka `matched: true` hai aur jinka `.wav` file physically maujood hai (kuch reciters ke paas basmalah segments nahi hain). Har entry mein `surah` field bhi hoti hai taake 113 aur 114 ke segments alag identify ho saken.

In [ ]:
import json
import glob
import os

# Dono surahs ki reference JSON dhoondo (kisi bhi reciter subfolder ke andar)
json_files = sorted(
    glob.glob(f'{DATASET_DIR}/**/surah_113_al-falaq__reference_segments.json', recursive=True)
    + glob.glob(f'{DATASET_DIR}/**/surah_114_an-nas__reference_segments.json', recursive=True)
)
print(f'{len(json_files)} reciter reference files mile (113 + 114).')

manifest = []
per_reciter_count = {}
per_surah_count = {}

for jf in json_files:
    reciter_dir = os.path.dirname(jf)
    with open(jf, 'r', encoding='utf-8') as f:
        data = json.load(f)

    reciter_name = data['qari']['name']
    surah_num = data['surah']['number']
    per_reciter_count.setdefault(reciter_name, 0)
    per_surah_count.setdefault(surah_num, 0)

    for seg in data['segments']:
        if not seg.get('matched'):
            continue
        wav_path = os.path.join(reciter_dir, seg['audio_segment_file'])
        if not os.path.exists(wav_path):
            continue

        manifest.append({
            'file': wav_path,
            'text': seg['transcription'],
            'reciter': reciter_name,
            'seg_id': f"s{surah_num}_{seg['id']}",  # unique across both surahs
            'surah': surah_num,
            'surah_ayah': seg.get('surah_ayah'),
        })
        per_reciter_count[reciter_name] += 1
        per_surah_count[surah_num] += 1

print(f'\nTotal usable segments: {len(manifest)}\n')
print('Surah-wise segment count:')
for num, cnt in sorted(per_surah_count.items()):
    print(f'  Surah {num}: {cnt} segments')
print('\nReciter-wise segment count:')
for name, cnt in sorted(per_reciter_count.items()):
    print(f'  {name:35s} -> {cnt} segments')

print('\nSample entries:')
for m in manifest[:5]:
    print(f"  [surah {m['surah']}] [{m['reciter']}] {m['seg_id']} -> {m['text']}")


## 5. Train / Validation split — reciter-level (real generalization test)

Pehle wale demo mein training aur testing **same 12 clips** par hoti thi — jo sirf memorization dikhata tha, generalization nahi. Ab hum **2 reciters ko poori tarah alag** rakhte hain validation ke liye — matlab model training ke doran un dono ki awaaz kabhi nahi sunta. Isse hum real accuracy dekh sakte hain: naye/unseen reciter par model kaisa perform karta hai.

(Agar aapke paas kam reciters bache to warning aayegi — dataset dobara check karein.)

In [ ]:
import random

random.seed(42)

all_reciters = sorted(per_reciter_count.keys())
print(f'Total reciters: {len(all_reciters)}')

# 2 reciters ko validation (unseen/held-out) ke liye alag rakhte hain
HELD_OUT_COUNT = 2
held_out_reciters = set(random.sample(all_reciters, min(HELD_OUT_COUNT, max(1, len(all_reciters) - 1))))

train_manifest = [m for m in manifest if m['reciter'] not in held_out_reciters]
val_manifest = [m for m in manifest if m['reciter'] in held_out_reciters]

print(f'\nHeld-out (unseen) reciters for validation: {sorted(held_out_reciters)}')
print(f'Training samples: {len(train_manifest)}')
print(f'Validation samples (unseen reciters): {len(val_manifest)}')

if len(val_manifest) == 0 or len(train_manifest) == 0:
    print('\n[WARNING] Split theek nahi bana — dataset ya HELD_OUT_COUNT dobara check karein.')


## 6. Base model aur processor load karein

In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_ID = 'tarteel-ai/whisper-base-ar-quran'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

processor = WhisperProcessor.from_pretrained(MODEL_ID, language='arabic', task='transcribe')
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.to(device)

print('Base model loaded.')

## 7. Dataset class banayein

Har audio file ko load karta hai, 16kHz mono mein convert karta hai (librosa se), aur processor ke through Whisper ke expected format (log-mel spectrogram + tokenized labels) mein convert karta hai. Ab ye full absolute path directly manifest se leta hai (reciter folders alag-alag hain isliye ek single `audio_dir` kaafi nahi tha).

In [ ]:
import librosa
from torch.utils.data import Dataset

class QuranSegmentDataset(Dataset):
    def __init__(self, manifest, processor):
        self.manifest = manifest
        self.processor = processor

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        item = self.manifest[idx]
        audio_array, sr = librosa.load(item['file'], sr=16000)

        input_features = self.processor(
            audio_array, sampling_rate=16000, return_tensors='pt'
        ).input_features[0]

        labels = self.processor.tokenizer(item['text']).input_ids

        return {'input_features': input_features, 'labels': labels}

train_dataset = QuranSegmentDataset(train_manifest, processor)
eval_dataset = QuranSegmentDataset(val_manifest, processor)

print(f'Train dataset: {len(train_dataset)} samples')
print(f'Eval dataset (unseen reciters): {len(eval_dataset)} samples')

sample = train_dataset[0]
print('input_features shape:', sample['input_features'].shape)
print('labels:', sample['labels'])

## 8. Data collator (batching ke liye padding)

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')

        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')

        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch['labels'] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print('Data collator ready.')

## 9. WER metric (compute_metrics)

Ab dataset bada hai to hum sirf "eyeball" match nahi karte — proper Word Error Rate (WER) metric use karte hain, jo held-out (unseen) reciters par model ki asal generalization measure karega.

In [ ]:
import evaluate

wer_metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {'wer': wer}

print('WER metric ready.')

## 10. Training arguments + Trainer setup

Dataset ab **~330 samples** ka hai (12 reciters mein se ~10 train ke liye, 2 held-out), pehle wale 12-sample demo se kaafi bada. Isliye:
- `max_steps` thoda badhaya gaya hai (dono surahs ka data)
- Periodic evaluation on held-out (unseen) reciters — taake training ke doran hi generalization track ho sake
- Phir bhi conservative learning rate rakha hai taake catastrophic forgetting na ho (model apni pehle ki general Quran knowledge na bhool jaye)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

OUTPUT_DIR = '/content/whisper-quran-finetuned'

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=20,
    max_steps=500,
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    predict_with_generate=True,
    logging_steps=10,
    report_to=[],
    remove_unused_columns=False,
    label_names=['labels'],
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

print('Trainer ready. Starting training...')

## 11. Training chalayein

In [ ]:
trainer.train()
print('Training complete.')

## 12. Fine-tuned model save karein

In [ ]:
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')

## 13. Test karein — train reciters vs unseen (held-out) reciters

**Ye hissa purani demo se sabse important farq hai.** Pehle hum sirf training clips par hi test karte thay, jo sirf "yaad kiya ya nahi" dikhata tha. Ab hum **do groups** alag test karte hain:

1. **Train reciters** (jinki awaaz model ne suni) — high accuracy expected hai
2. **Held-out/unseen reciters** (jinki awaaz model ne kabhi nahi suni) — ye asal generalization ka proof hai

Agar unseen reciters par bhi accuracy achi hai to matlab model ne actually kuch seekha hai, sirf memorize nahi kiya.

In [ ]:
model.eval()

import re

# Whisper control tokens (<|startoftranscript|>, <|ar|>, <|transcribe|>, <|notimestamps|>, etc.)
# kabhi kabhi skip_special_tokens=True inhe strip nahi karta (fine-tuned model ke saath
# newer transformers versions mein ye dekha gaya hai) — isliye regex se safai bhi kar dete hain.
def clean_text(text):
    text = re.sub(r'<\|[^|]*\|>', '', text)
    return text.strip()

def run_inference(entries, label):
    correct = 0
    print(f'\n=== {label} ({len(entries)} samples) ===')
    for item in entries:
        audio_array, sr = librosa.load(item['file'], sr=16000)
        inputs = processor(audio_array, sampling_rate=16000, return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            generated_ids = model.generate(inputs['input_features'])

        raw_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        predicted_text = clean_text(raw_text)
        expected_text = clean_text(item['text'])

        match = predicted_text == expected_text
        correct += int(match)
        mark = '\u2705' if match else '\u274c'
        print(f"{mark} [{item['reciter'][:20]:20s}] {item['seg_id']:8s} | Expected: {expected_text:30s} | Got: {predicted_text}")

    acc = 100 * correct / max(1, len(entries))
    print(f'\n{label} exact-match accuracy: {acc:.1f}% ({correct}/{len(entries)})')
    return acc

train_acc = run_inference(train_manifest, 'TRAIN reciters (seen during training)')
val_acc = run_inference(val_manifest, 'HELD-OUT reciters (unseen \u2014 real generalization test)')

print(f'\nSummary: Train accuracy = {train_acc:.1f}%  |  Unseen-reciter accuracy = {val_acc:.1f}%')
print('Bara gap (train >> unseen) matlab overfitting; close numbers matlab real generalization.')


## 14. Model download karein (zip banayein)

In [ ]:
import shutil
shutil.make_archive('/content/whisper-quran-finetuned', 'zip', OUTPUT_DIR)

from google.colab import files
files.download('/content/whisper-quran-finetuned.zip')